# Notebook 21 — MJO NSV Stage 0 + Stage 1 (Bandpassed All-Year Data, lag = 10)
**Project:** ENSO-BSISO SSL — MJO NSV extension  
**Author:** Jiayi (jh9141@nyu.edu)

Combined Stage 0 (data prep) + Stage 1 (encoder-decoder training) for MJO NSV. Parallel to BSISO's `nb17b` + `nb18c` but with MJO-specific adjustments. See Session 34 of `results/conversation_log.md` for the full plan.

## Method (one-paragraph summary)

Train a CNN encoder `g_E` + decoder `g_D` to predict the atmospheric field 10 days ahead:

$$\mathcal{L}_1 = \mathbb{E}_{(X_t, X_{t+10})}\bigl[\,\lVert g_D(g_E(X_t)) - X_{t+10} \rVert^2\,\bigr]$$

Overparameterized 64-D bottleneck (true MJO ID expected 2–4). The 10-day lag forces the encoder to extract the slow MJO state — at lag-1 persistence would be trivially good and the encoder would degenerate to a lossy autoencoder (the BSISO nb18b failure mode, Session 31).

## Inputs (MJO/data/processed/)

- `X_MJO_bp20_90.npy` — shape `(N≈16,245, 3, 1, 180)`. Lee preprocessing + Lanczos 20–90 d bandpass (the same file `nb15` used). Channels: **`[u850, OLR, u200]`** (different order than BSISO).
- `labels_aligned_mjo_bp20_90.csv` — RMM phase (1–8), RMM amplitude, ENSO category (El Nino/Neutral/La Nina), `weak_mjo` boolean, date.

## Key MJO-vs-BSISO differences applied here

| | BSISO (nb18c) | MJO (this notebook) |
|---|---|---|
| Spatial shape | `(3, 31, 51)` 2-D | `(3, 180)` 1-D (singleton lat squeezed) |
| Channels | `[u850, v850, OLR]` | `[u850, OLR, u200]` |
| Temporal scope | MJJAS only | **All-year** |
| Bandpass | lp25 (25-d lowpass) | **bp20–90** (already removes annual cycle) |
| Pair rule | `(δ=10) & same_year` | `(δ=10) & same_split` (no winter gap to exclude) |
| Architecture | 2-D Conv (5 blocks) | **1-D Conv** (5 blocks) |
| N pairs (lag-10) | ~3,999 | **~15,500** (5× more) |
| ENSO baseline | nb05 sup z = 11.02 | nb14 sup z = 12.21, nb16 RMM z = 4.10 |

## Outputs (`MJO/nsv/...`)

```
MJO/nsv/data_lag10/
  X_t.npy, X_t1.npy            — lag-10 pair tensors
  dates_t.npy                  — anchor date for each pair
  rmm_phase_t.npy              — RMM phase 1..8 aligned to anchors
  rmm_amplitude_t.npy          — RMM amplitude aligned to anchors
  enso_cat_t.npy               — ENSO category (string)
  weak_mjo_t.npy               — boolean: True when amp < 1 (for analysis filter in nb23)
  train_mask.npy               — boolean train/val split (every 5th year held out)
  nsv_data_meta.json           — pair counts + split summary
MJO/nsv/checkpoints_lag10/
  encoder_stage1{,_best}.pth, decoder_stage1{,_best}.pth
  training_history_stage1.json
MJO/nsv/latents_lag10/
  z_train.npy, z_val.npy
  (label arrays copied here for nb22's auto-load)
MJO/nsv/results/stage1_lag10/
  training_curves.png, reconstructions.png, latent_diagnostics.png
  stage1_summary.json
```

## Verification gate (must pass before nb22)

1. **Beats persistence**: val MSE < lag-10 persistence MSE × 0.9 (i.e., +10% improvement minimum).
2. **PC1 > 25%** on z_train PCA — the manifold-structure target.
3. **Reconstructions** look spatially coherent (not blurred constants).

If any fails, retry with lag=15 (then lag=20). If all lag values fail → MJO NSV falsified, document as null result.

## Runtime

~5–8 min on Colab T4 (more data than BSISO, but smaller 1-D model).

---

## Cell 1 — Setup: Paths, Hyperparameters, Load Raw Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

PROJECT_DIR   = '/content/drive/MyDrive/BSISO_SSL_Project'
MJO_DIR       = f'{PROJECT_DIR}/MJO'
PROCESSED_DIR = f'{MJO_DIR}/data/processed'

NSV_DIR      = f'{MJO_DIR}/nsv'
DATA_DIR     = f'{NSV_DIR}/data_lag10'
CKPT_DIR     = f'{NSV_DIR}/checkpoints_lag10'
LATENT_DIR   = f'{NSV_DIR}/latents_lag10'
RESULTS_DIR  = f'{NSV_DIR}/results/stage1_lag10'
for d in [DATA_DIR, CKPT_DIR, LATENT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# Hyperparameters — match BSISO nb18c except for the architecture (1-D for MJO)
LAG_DAYS       = 10
LATENT_DIM     = 64
BATCH_SIZE     = 64
EPOCHS         = 100
LR             = 1e-3
WEIGHT_DECAY   = 1e-4
SEED           = 42
VAL_YEARS_STRIDE = 5

torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device:     {device}')
print(f'Lag:        {LAG_DAYS} days')
print(f'Latent dim: {LATENT_DIM}')

# Load the bandpassed MJO data
X_FILE      = 'X_MJO_bp20_90.npy'
LABELS_FILE = 'labels_aligned_mjo_bp20_90.csv'

X      = np.load(f'{PROCESSED_DIR}/{X_FILE}')
labels = pd.read_csv(f'{PROCESSED_DIR}/{LABELS_FILE}', parse_dates=['date'])
labels['date'] = labels['date'].dt.normalize()

assert X.shape[0] == len(labels), f'X / labels length mismatch: {X.shape[0]} vs {len(labels)}'
assert X.shape[1:] == (3, 1, 180), f'Unexpected X shape: {X.shape[1:]}; expected (3, 1, 180)'
print(f'\nLoaded {X.shape[0]} bandpassed days  ({labels["date"].min().date()} to {labels["date"].max().date()}).')
print(f'Channels: [u850, OLR, u200]   (MJO convention from nb13)')

## Cell 2 — Build Lag-10 Pairs + Year-Based Train/Val Split

MJO data is **continuous all-year** (no winter gap to exclude). So the BSISO `same_year` constraint doesn't apply — Dec 31 → Jan 1 is a normal 1-day transition. The replacement rule is `same_split(anchor, target)`: a pair is valid iff its anchor and target belong to the same train/val split (excludes leakage but allows all real consecutive-day pairs).

Also: squeeze the singleton lat dim from `(N, 3, 1, 180)` → `(N, 3, 180)` for downstream 1-D Conv processing.

In [ ]:
# Defensive sort
order = np.argsort(labels['date'].values)
if not np.array_equal(order, np.arange(len(labels))):
    X = X[order]
    labels = labels.iloc[order].reset_index(drop=True)

# Squeeze singleton lat
X = X.reshape(X.shape[0], 3, 180).astype(np.float32)
print(f'X reshaped to {X.shape} (singleton lat squeezed).')

dates_all = pd.DatetimeIndex(labels['date'].values)
years_all = dates_all.year.values

# Year-based train/val split (every 5th year held out — matches nb14/nb15)
all_years   = sorted(np.unique(years_all).tolist())
val_years   = all_years[::VAL_YEARS_STRIDE]
train_years = sorted(set(all_years) - set(val_years))
print(f'\nVal years ({len(val_years)}): {val_years}')
print(f'Train years ({len(train_years)})')

# Build lag-LAG pair indices with same-split rule (not same-year — MJO is continuous)
delta_days     = (dates_all[LAG_DAYS:] - dates_all[:-LAG_DAYS]).days
in_val_anchor  = np.isin(years_all[:-LAG_DAYS], val_years)
in_val_target  = np.isin(years_all[LAG_DAYS:],  val_years)
no_leakage     = in_val_anchor == in_val_target
valid_start    = (delta_days == LAG_DAYS) & no_leakage
valid_start    = np.concatenate([valid_start, np.zeros(LAG_DAYS, dtype=bool)])

pair_idx_t   = np.where(valid_start)[0]
pair_idx_t1  = pair_idx_t + LAG_DAYS
N_pairs      = len(pair_idx_t)

# Hard verification
actual_deltas = (dates_all[pair_idx_t1] - dates_all[pair_idx_t]).days
assert actual_deltas.min() == LAG_DAYS and actual_deltas.max() == LAG_DAYS
leak_anchor_val = np.isin(years_all[pair_idx_t],  val_years)
leak_target_val = np.isin(years_all[pair_idx_t1], val_years)
assert (leak_anchor_val == leak_target_val).all(), 'Train/val leakage pair leaked through!'
print(f'\nPair construction:  {N_pairs} valid lag-{LAG_DAYS} pairs  (expected ~15,500).')

# Build pair tensors and label arrays aligned to anchors
X_t              = X[pair_idx_t]
X_t1             = X[pair_idx_t1]
dates_t          = dates_all[pair_idx_t]
rmm_phase_t      = labels['phase'].values[pair_idx_t].astype(np.int8)
rmm_amplitude_t  = labels['amplitude'].values[pair_idx_t].astype(np.float32)
enso_cat_t       = labels['enso_category'].values[pair_idx_t].astype('<U10')
weak_mjo_t       = labels['weak_mjo'].values[pair_idx_t].astype(bool)

# Train/val mask (by year of anchor)
pair_years = dates_t.year.values
train_mask = np.isin(pair_years, train_years)
n_train = int(train_mask.sum()); n_val = int((~train_mask).sum())
print(f'Train: {n_train} pairs ({100*n_train/N_pairs:.1f}%) from {len(train_years)} years')
print(f'Val:   {n_val} pairs ({100*n_val/N_pairs:.1f}%) from {len(val_years)} years')

# Active-MJO fraction per split (informational; used by nb23)
print(f'\nActive-MJO fraction (amp >= 1):')
for split_name, mask in [('train', train_mask), ('val', ~train_mask)]:
    w = weak_mjo_t[mask]
    print(f'  {split_name:<6}: {int((~w).sum())}/{int(mask.sum())} ({100*(~w).mean():.1f}% active)')

# Save data files
np.save(f'{DATA_DIR}/X_t.npy', X_t)
np.save(f'{DATA_DIR}/X_t1.npy', X_t1)
np.save(f'{DATA_DIR}/dates_t.npy', dates_t.values.astype('datetime64[D]'))
np.save(f'{DATA_DIR}/rmm_phase_t.npy', rmm_phase_t)
np.save(f'{DATA_DIR}/rmm_amplitude_t.npy', rmm_amplitude_t)
np.save(f'{DATA_DIR}/enso_cat_t.npy', enso_cat_t)
np.save(f'{DATA_DIR}/weak_mjo_t.npy', weak_mjo_t)
np.save(f'{DATA_DIR}/train_mask.npy', train_mask)

# Also save label arrays under latents_lag10 so nb22 can auto-detect (matches BSISO nb18c convention)
np.save(f'{LATENT_DIR}/dates_t.npy', dates_t.values.astype('datetime64[D]'))
np.save(f'{LATENT_DIR}/rmm_phase_t.npy', rmm_phase_t)
np.save(f'{LATENT_DIR}/rmm_amplitude_t.npy', rmm_amplitude_t)
np.save(f'{LATENT_DIR}/enso_cat_t.npy', enso_cat_t)
np.save(f'{LATENT_DIR}/weak_mjo_t.npy', weak_mjo_t)
np.save(f'{LATENT_DIR}/train_mask.npy', train_mask)

meta = {
    'source_X':         X_FILE,
    'source_labels':    LABELS_FILE,
    'preprocessing':    'Lee MJO + Lanczos 20-90 d bandpass (removes synoptic noise AND annual cycle)',
    'channels':         ['u850', 'OLR', 'u200'],
    'spatial_shape':    [3, 180],
    'temporal_scope':   'all-year, continuous (1979-2023)',
    'pair_rule':        f'(delta_days == {LAG_DAYS}) & same_split(anchor, target)',
    'lag_days':         int(LAG_DAYS),
    'n_pairs_total':    int(N_pairs),
    'n_pairs_train':    int(n_train),
    'n_pairs_val':      int(n_val),
    'train_years':      [int(y) for y in train_years],
    'val_years':        [int(y) for y in val_years],
    'val_year_stride':  int(VAL_YEARS_STRIDE),
    'date_min':         str(dates_t.min().date()),
    'date_max':         str(dates_t.max().date()),
    'rmm_phase_col':    'phase',
    'rmm_amplitude_col': 'amplitude',
}
with open(f'{DATA_DIR}/nsv_data_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print(f'\nSaved data + label arrays to {DATA_DIR} and {LATENT_DIR}')

## Cell 3 — 1-D Encoder + Decoder Architecture

Input: `(B, 3, 180)`. Five Conv1d blocks (3 stride-2) compress to `(128, 22)`, then `AdaptiveAvgPool1d(1)` + Linear → `z ∈ ℝ^{64}`.

Decoder: Linear → reshape `(128, 1)`, then 5 stages of bilinear-1D `F.interpolate` + Conv1d to hit exactly 180 lon. The interp-to-exact-size pattern is the same trick BSISO's decoder used for the non-power-of-2 `(31, 51)` target.

Total ~107 K params (smaller than BSISO's 230 K because the spatial dim is 1-D).

In [ ]:
class EncoderMJO(nn.Module):
    """1-D CNN encoder for MJO: (B, 3, 180) -> (B, latent_dim)."""
    def __init__(self, latent_dim=64):
        super().__init__()
        self.conv1 = nn.Conv1d(3,   16,  kernel_size=4, stride=2, padding=1, bias=False)  # 180 -> 90
        self.bn1   = nn.BatchNorm1d(16)
        self.conv2 = nn.Conv1d(16,  32,  kernel_size=3, stride=1, padding=1, bias=False)  # 90 -> 90
        self.bn2   = nn.BatchNorm1d(32)
        self.conv3 = nn.Conv1d(32,  32,  kernel_size=4, stride=2, padding=1, bias=False)  # 90 -> 45
        self.bn3   = nn.BatchNorm1d(32)
        self.conv4 = nn.Conv1d(32,  64,  kernel_size=3, stride=1, padding=1, bias=False)  # 45 -> 45
        self.bn4   = nn.BatchNorm1d(64)
        self.conv5 = nn.Conv1d(64,  128, kernel_size=4, stride=2, padding=1, bias=False)  # 45 -> 22
        self.bn5   = nn.BatchNorm1d(128)
        self.gap   = nn.AdaptiveAvgPool1d(1)
        self.fc    = nn.Linear(128, latent_dim)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01); nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = F.relu(self.bn5(self.conv5(x)))
        x = self.gap(x).flatten(1)
        return self.fc(x)


class DecoderMJO(nn.Module):
    """1-D bilinear-upsample + Conv1d decoder: (B, latent_dim) -> (B, 3, 180)."""
    def __init__(self, latent_dim=64):
        super().__init__()
        self.fc    = nn.Linear(latent_dim, 128)
        self.conv1 = nn.Conv1d(128, 64,  kernel_size=3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm1d(64)
        self.conv2 = nn.Conv1d(64,  64,  kernel_size=3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm1d(64)
        self.conv3 = nn.Conv1d(64,  32,  kernel_size=3, padding=1, bias=False)
        self.bn3   = nn.BatchNorm1d(32)
        self.conv4 = nn.Conv1d(32,  16,  kernel_size=3, padding=1, bias=False)
        self.bn4   = nn.BatchNorm1d(16)
        self.conv5 = nn.Conv1d(16,  3,   kernel_size=3, padding=1)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None: nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01); nn.init.constant_(m.bias, 0)

    def forward(self, z):
        x = self.fc(z).view(-1, 128, 1)
        x = F.interpolate(x, size=3,   mode='linear', align_corners=False); x = F.relu(self.bn1(self.conv1(x)))
        x = F.interpolate(x, size=12,  mode='linear', align_corners=False); x = F.relu(self.bn2(self.conv2(x)))
        x = F.interpolate(x, size=45,  mode='linear', align_corners=False); x = F.relu(self.bn3(self.conv3(x)))
        x = F.interpolate(x, size=90,  mode='linear', align_corners=False); x = F.relu(self.bn4(self.conv4(x)))
        x = F.interpolate(x, size=180, mode='linear', align_corners=False)
        return self.conv5(x)


enc = EncoderMJO(LATENT_DIM).to(device)
dec = DecoderMJO(LATENT_DIM).to(device)
n_params = sum(p.numel() for p in enc.parameters()) + sum(p.numel() for p in dec.parameters())
with torch.no_grad():
    dummy = torch.randn(4, 3, 180).to(device)
    z = enc(dummy); xhat = dec(z)
    assert z.shape == (4, LATENT_DIM) and xhat.shape == dummy.shape, f'Shape mismatch: z={z.shape}, xhat={xhat.shape}'
    print(f'Total params: {n_params:,}  (expected ~107K, smaller than BSISO 230K due to 1-D)')
    print(f'Forward sanity:  input {tuple(dummy.shape)}  ->  z {tuple(z.shape)}  ->  xhat {tuple(xhat.shape)}')
    print(f'Init MSE on random input:  {F.mse_loss(xhat, dummy).item():.4f}')
    print('✓ Architecture sanity-checked.')

## Cell 4 — Dataset + DataLoaders + Persistence Baseline

In [ ]:
class PairDataset(Dataset):
    def __init__(self, X_t, X_t1, indices):
        self.X_t  = torch.from_numpy(X_t[indices]).float()
        self.X_t1 = torch.from_numpy(X_t1[indices]).float()
    def __len__(self):  return self.X_t.shape[0]
    def __getitem__(self, k):  return self.X_t[k], self.X_t1[k]

train_idx_arr = np.where(train_mask)[0]
val_idx_arr   = np.where(~train_mask)[0]
train_ds = PairDataset(X_t, X_t1, train_idx_arr)
val_ds   = PairDataset(X_t, X_t1, val_idx_arr)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {len(train_ds):5d} pairs  → {len(train_loader)} batches/epoch')
print(f'Val:   {len(val_ds):5d} pairs  → {len(val_loader)} batches/epoch')

# Persistence baseline at lag-LAG (predict X_{t+10} = X_t)
with torch.no_grad():
    persistence_mse = ((val_ds.X_t - val_ds.X_t1) ** 2).mean().item()
print(f'\nPersistence MSE at lag={LAG_DAYS} (val): {persistence_mse:.4f}')
print(f'  Model must beat this by at least 10%.')
print(f'  BSISO nb18c baseline at lag=10: 0.295  (model achieved 0.239 → +18.9%)')

## Cell 5 — Training Loop

100 epochs, Adam lr=1e-3, CosineAnnealingLR. Same recipe as BSISO nb18c. Save best-val checkpoint; final-epoch weights also kept.

In [ ]:
from tqdm.notebook import tqdm

params = list(enc.parameters()) + list(dec.parameters())
optimizer = optim.Adam(params, lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR * 0.01)

history = {'train_mse': [], 'val_mse': [], 'epoch_time': []}
best_val = float('inf')
t_start = time.time()

for epoch in range(EPOCHS):
    t0 = time.time()
    enc.train(); dec.train()
    train_loss = 0.0; n_seen = 0
    pbar = tqdm(train_loader, desc=f'ep {epoch+1}/{EPOCHS}', leave=False)
    for x_t, x_t1 in pbar:
        x_t  = x_t.to(device, non_blocking=True); x_t1 = x_t1.to(device, non_blocking=True)
        loss = F.mse_loss(dec(enc(x_t)), x_t1)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        train_loss += loss.item() * x_t.size(0); n_seen += x_t.size(0)
        pbar.set_postfix({'mse': f'{loss.item():.4f}'})
    train_mse = train_loss / n_seen

    enc.eval(); dec.eval()
    val_loss = 0.0; n_seen = 0
    with torch.no_grad():
        for x_t, x_t1 in val_loader:
            x_t  = x_t.to(device, non_blocking=True); x_t1 = x_t1.to(device, non_blocking=True)
            val_loss += F.mse_loss(dec(enc(x_t)), x_t1, reduction='sum').item() / x_t1.numel() * x_t.size(0)
            n_seen += x_t.size(0)
    val_mse = val_loss / n_seen
    scheduler.step()

    et = time.time() - t0
    history['train_mse'].append(train_mse); history['val_mse'].append(val_mse); history['epoch_time'].append(et)
    if (epoch + 1) % 10 == 0 or epoch < 3:
        print(f'ep {epoch+1:3d}/{EPOCHS}   train={train_mse:.4f}   val={val_mse:.4f}   '
              f'lr={scheduler.get_last_lr()[0]:.2e}   time={et:.1f}s')

    if val_mse < best_val:
        best_val = val_mse
        torch.save(enc.state_dict(), f'{CKPT_DIR}/encoder_stage1_best.pth')
        torch.save(dec.state_dict(), f'{CKPT_DIR}/decoder_stage1_best.pth')

torch.save(enc.state_dict(), f'{CKPT_DIR}/encoder_stage1.pth')
torch.save(dec.state_dict(), f'{CKPT_DIR}/decoder_stage1.pth')
with open(f'{CKPT_DIR}/training_history_stage1.json', 'w') as f:
    json.dump(history, f, indent=2)

print(f'\nDone in {(time.time()-t_start)/60:.1f} min.')
print(f'Best val MSE:    {best_val:.4f}  (at epoch {history["val_mse"].index(best_val)+1})')
print(f'Final train MSE: {history["train_mse"][-1]:.4f}   val MSE: {history["val_mse"][-1]:.4f}')
print(f'Persistence (lag={LAG_DAYS}): {persistence_mse:.4f}')
imp = (persistence_mse - best_val) / persistence_mse * 100
print(f'Improvement over persistence: {imp:+.1f}%  (target > +10%)')

## Cell 6 — Diagnostics + Extract Latents

Three figures + latent extraction:
1. **Training curves** + per-epoch time
2. **Reconstructions** — 4 random val pairs as longitude profiles (OLR channel)
3. **Latent diagnostics** — per-dim std + PCA scree. **The headline number is PC1 % on the scree subplot** — if > 25%, manifold structure is real and nb22 can run.

In [ ]:
enc.load_state_dict(torch.load(f'{CKPT_DIR}/encoder_stage1_best.pth', map_location=device))
dec.load_state_dict(torch.load(f'{CKPT_DIR}/decoder_stage1_best.pth', map_location=device))
enc.eval(); dec.eval()

# 1) Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(history['train_mse'], label='Train MSE', lw=2)
axes[0].plot(history['val_mse'],   label='Val MSE',   lw=2)
axes[0].axhline(persistence_mse, color='gray', ls='--', lw=1, label=f'Persistence ({persistence_mse:.3f})')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE')
axes[0].set_title(f'MJO Stage 1 training  (bp20-90, lag={LAG_DAYS})', fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history['epoch_time'], color='green', lw=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Time (s)')
axes[1].set_title('Per-Epoch Time', fontweight='bold')
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/training_curves.png', dpi=140, bbox_inches='tight'); plt.show()

# 2) Reconstructions on 4 random val pairs (OLR channel — channel 1 in MJO order [u850, OLR, u200])
rng = np.random.default_rng(SEED)
picks = rng.choice(len(val_ds), size=4, replace=False)
x_t_pick  = val_ds.X_t[picks].to(device); x_t1_pick = val_ds.X_t1[picks].to(device)
with torch.no_grad():
    x_t1_hat = dec(enc(x_t_pick)).cpu().numpy()
x_t_np  = x_t_pick.cpu().numpy(); x_t1_np = x_t1_pick.cpu().numpy()
lons    = np.arange(0, 360, 2)   # MJO: 0-358°E at 2°

olr_ch = 1
fig, axes = plt.subplots(4, 1, figsize=(12, 11), sharex=True)
fig.suptitle(f"MJO val reconstructions  (OLR longitude profile, lag={LAG_DAYS})", fontsize=12, fontweight='bold')
for r in range(4):
    ax = axes[r]
    ax.plot(lons, x_t_np[r, olr_ch],   color='gray',     lw=1.5, alpha=0.7, label='X_t (today)')
    ax.plot(lons, x_t1_np[r, olr_ch],  color='steelblue', lw=2,   label=f'X_t+{LAG_DAYS} target')
    ax.plot(lons, x_t1_hat[r, olr_ch], color='red',       lw=2, ls='--', label=f'X_t+{LAG_DAYS} predicted')
    ax.set_ylabel(f"OLR' (σ)\npair {picks[r]}", fontsize=10)
    ax.axhline(0, color='k', lw=0.4, alpha=0.4)
    ax.grid(alpha=0.3)
    if r == 0: ax.legend(fontsize=9, loc='best')
axes[-1].set_xlabel('Longitude (°)')
plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/reconstructions.png', dpi=130, bbox_inches='tight'); plt.show()

# 3) Latent diagnostics — per-dim std + PCA scree
with torch.no_grad():
    z_train_diag = np.concatenate([enc(train_ds.X_t[k:k+256].to(device)).cpu().numpy()
                                    for k in range(0, len(train_ds), 256)], axis=0)
z_std = z_train_diag.std(axis=0)
n_active = int((z_std > 0.01).sum())
from sklearn.decomposition import PCA
pca_diag = PCA(n_components=min(15, LATENT_DIM)).fit(z_train_diag - z_train_diag.mean(0))
var_ratio_diag = pca_diag.explained_variance_ratio_

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
ax = axes[0]
ax.bar(np.arange(LATENT_DIM), z_std, color='steelblue', alpha=0.85)
ax.axhline(z_std.mean(), color='red', ls='--', lw=1, label=f'Mean = {z_std.mean():.3f}')
ax.axhline(0.01, color='gray', ls=':', lw=1, label='Collapse threshold')
ax.set_xlabel('Latent dim'); ax.set_ylabel('std across train pairs')
ax.set_title(f'MJO per-dim std  (active: {n_active}/{LATENT_DIM})', fontweight='bold', fontsize=11)
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.bar(range(1, len(var_ratio_diag)+1), var_ratio_diag*100, color='steelblue', alpha=0.85)
ax.set_xlabel('PC index'); ax.set_ylabel('% variance')
ax.set_title(f'MJO PCA scree (first {len(var_ratio_diag)} PCs)\n'
             f'PC1 = {var_ratio_diag[0]*100:.1f}%   '
             f'(BSISO nb18c was 85.8%; target > 25% for real manifold)',
             fontweight='bold', fontsize=11)
ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/latent_diagnostics.png', dpi=140, bbox_inches='tight'); plt.show()

print(f'\nLatent diagnostics (MJO bp20-90, lag={LAG_DAYS}):')
print(f'  active dims:  {n_active}/{LATENT_DIM}')
print(f'  per-dim std:  min={z_std.min():.4f}, max={z_std.max():.4f}, mean={z_std.mean():.4f}')
print(f'  PC1 / PC2:    {var_ratio_diag[0]*100:.2f}%  /  {var_ratio_diag[1]*100:.2f}%')
print(f'  cum at 5 PCs: {np.cumsum(var_ratio_diag)[4]*100:.2f}%')
if var_ratio_diag[0] > 0.25:
    print('✓ PC1 > 25% — manifold appears. Proceed to nb22 (ID estimation).')
elif var_ratio_diag[0] > 0.15:
    print('△ PC1 15-25% — partial manifoldization. nb22 may give MEDIUM confidence.')
else:
    print('✗ PC1 < 15% — manifold diffuse. Consider lag=15 retry.')

# 4) Extract & save full-set latents for nb22
def extract_z(dataset, encoder, batch=256):
    encoder.eval()
    with torch.no_grad():
        return np.concatenate([encoder(dataset.X_t[k:k+batch].to(device)).cpu().numpy()
                                for k in range(0, len(dataset), batch)], axis=0).astype(np.float32)
z_train = extract_z(train_ds, enc)
z_val   = extract_z(val_ds,   enc)
np.save(f'{LATENT_DIR}/z_train.npy', z_train)
np.save(f'{LATENT_DIR}/z_val.npy',   z_val)
print(f'\nSaved latents: z_train {z_train.shape}, z_val {z_val.shape}  →  {LATENT_DIR}/')

import time as _t
stage1_summary = {
    'date':                _t.strftime('%Y-%m-%d'),
    'pipeline':            'MJO NSV',
    'preprocessing':       'bp20-90',
    'lag_days':            LAG_DAYS,
    'latent_dim':          LATENT_DIM,
    'epochs':              EPOCHS,
    'n_pairs_total':       int(N_pairs),
    'n_pairs_train':       int(n_train),
    'n_pairs_val':         int(n_val),
    'best_val_mse':        float(best_val),
    'final_train_mse':     float(history['train_mse'][-1]),
    'final_val_mse':       float(history['val_mse'][-1]),
    'persistence_mse':     float(persistence_mse),
    'improvement_pct':     float(imp),
    'n_active_dims':       int(n_active),
    'pc1_var_ratio':       float(var_ratio_diag[0]),
    'pc2_var_ratio':       float(var_ratio_diag[1]),
    'cum_var_5pcs':        float(np.cumsum(var_ratio_diag)[4]),
    'n_params':            int(n_params),
    'meta_from_data':      meta,
}
with open(f'{RESULTS_DIR}/stage1_summary.json', 'w') as f:
    json.dump(stage1_summary, f, indent=2)
print(f'\nSaved summary: {RESULTS_DIR}/stage1_summary.json')

---
## Done!

**Send back** for review:
1. Cell 5 final printed lines — best val MSE + persistence + improvement %.
2. `results/stage1_lag10/training_curves.png`.
3. `results/stage1_lag10/latent_diagnostics.png` — **the headline figure**; specifically the PC1 % on the scree subplot.
4. `results/stage1_lag10/reconstructions.png` — should look spatially coherent.
5. `results/stage1_lag10/stage1_summary.json`.

**Decision tree based on PC1 %:**

| PC1 % | Action |
|---|---|
| > 25 | ✓ Manifold appeared. Proceed to nb22 (ID estimation). |
| 15–25 | △ Partial. Proceed to nb22 but expect MEDIUM confidence. |
| < 15 | ✗ Diffuse. Retry with lag=15 (set `LAG_DAYS = 15` at top of Cell 1). |

**Expected per Session 34 §1**:
- BSISO nb18c at lag=10 → PC1 = 85.8%
- MJO bp20-90 has the annual cycle already removed, so PC1 may be lower than BSISO if the actual MJO manifold is more isotropic / less dominated by one direction (consistent with RMM being 2-D by construction).
- PC1 in 40-80% range would be a healthy signal that the manifold is real but possibly 2-D-dominant (consistent with H1 or H2).

---
*DDCS Project | jh9141@nyu.edu*